# Okada Manila AIDA Survey Analysis and Objective Report

**Prepared by: Engr. Jamie Eduardo Rosal, MSCpE**  
**Dataset:** `[Capstone] Okada Manila UAI Survey (Responses).xlsx`  
**Main sample:** Qualified family respondents with children, expected `n = 229`

This notebook cleans and prepares the Okada Manila UAI survey dataset, computes an AIDA index for Attention, Interest, Desire, and Action, produces descriptive statistics and visualizations, and presents a third-person objective report aligned with the marketing plan for positioning Okada Manila as the Ultimate Family Destination in the Philippines.


In [ ]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
try:
    from IPython.display import display, HTML
except Exception:
    HTML = lambda x: x

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_MATPLOTLIB_SEABORN = True
except Exception:
    plt = None
    sns = None
    HAS_MATPLOTLIB_SEABORN = False

WORKBOOK = "[Capstone] Okada Manila UAI Survey (Responses).xlsx"
SHEET = "Form Responses 1"
AUTHOR = "Engr. Jamie Eduardo Rosal, MSCpE"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


## 1. Load the Workbook and Define Survey Aliases


In [ ]:
df = pd.read_excel(WORKBOOK, sheet_name=SHEET)
cols = list(df.columns)

alias = {
    "timestamp": cols[0], "score": cols[1], "consent": cols[2],
    "q1_children": cols[3], "q2_child_count": cols[4], "q3_child_ages": cols[5],
    "q4_family_situation": cols[6], "q5_staycation_behavior": cols[7],
    "q6_amenities": cols[8], "q7_activities": cols[9],
    "q8_family_amenity_considered": cols[10], "q9_activity_interest": cols[11],
    "q10_event_awareness": cols[12], "q11_promo_channels": cols[13],
    "q12_themed_experience": cols[14], "q13_staycation_purpose": cols[15],
    "q14_amenity_importance": cols[16], "q15_booking_motivators": cols[17],
    "q16_emotional_appeal": cols[18], "q17_budget_general": cols[19],
    "q18_event_participation": cols[20], "q19_event_driver": cols[21],
    "q20_package_interest": cols[22], "q21_package_book": cols[23],
    "q22_family_friendly_luxury_likelihood": cols[24], "q23_ir_awareness": cols[25],
    "q24_ir_preference": cols[26], "q25_ir_choice_reason": cols[27],
    "q26_okada_familiarity": cols[28], "q27_okada_sources": cols[29],
    "q28_competitor": cols[30], "q29_okada_offerings": cols[31],
    "q30_stayed_okada": cols[32], "q31a_booking_method_past": cols[33],
    "q31b_past_visits": cols[34], "q31c_planning_staycation": cols[35],
    "q31d_booking_method_planned": cols[36], "q31e_planned_visits": cols[37],
    "q32_stay_length": cols[38], "q33_room_type": cols[39],
    "q34_okada_budget": cols[40], "q35_booking_factor": cols[41],
    "q36_okada_family_package": cols[42], "q37_return": cols[43],
    "q37_events_interest": cols[44], "q37_experiential_interest": cols[45],
    "nickname": cols[46], "age": cols[47], "gender": cols[48],
    "nationality": cols[49], "education": cols[50], "occupation": cols[51],
    "income": cols[52], "location": cols[53], "social_platform": cols[54],
    "gcash": cols[55],
}

data_dictionary = pd.DataFrame(
    [{"Alias": k, "Original column": v.replace("\n", " ")} for k, v in alias.items()]
)
display(data_dictionary)


Alias,Original column
timestamp,Timestamp
score,Score
consent,Confidentiality and Data Privacy
q1_children,Q1. Do you have children?
q2_child_count,Q2. How many children do you have?
q3_child_ages,"Q3. What are the ages of your children? If you answered 2 or more in the previous question, check all boxes applicable according to their age brackets"
q4_family_situation,Q4. Which best describes your current family situation?
q5_staycation_behavior,Q5. Which statement best describes your family’s hotel or staycation behavior?
q6_amenities,"Q6. When planning to go out as a family, which amenities / activities do you usually look for? (Check all that apply)"
q7_activities,"Q7. When visiting a hotel or resort with your family, what activities do you usually do? (Check all that apply)"


## 2. Data Cleaning and Qualification


In [ ]:
qualified = df[df[alias["q1_children"]].eq("Yes")].copy()

identifier_columns = [alias["score"], alias["nickname"], alias["gcash"]]
analysis_df = qualified.drop(columns=identifier_columns)

cleaning_summary = pd.DataFrame([
    ["Raw responses loaded", len(df)],
    ["Raw columns loaded", len(df.columns)],
    ["Qualified respondents with children (main sample)", len(qualified)],
    ["Excluded rows without Q1 = Yes", len(df) - len(qualified)],
    ["Qualified rows with no consent response", (qualified[alias["consent"]] == "No, I do not give my consent.").sum()],
    ["Exact duplicate rows in qualified sample", qualified.duplicated().sum()],
], columns=["Check", "Result"])

missing_summary = (
    qualified.isna().sum().sort_values(ascending=False).head(18)
    .reset_index().rename(columns={"index": "Column", 0: "Missing values"})
)
missing_summary["Missing percent"] = (missing_summary["Missing values"] / len(qualified) * 100).round(1)

display(cleaning_summary)
display(missing_summary)


229 Qualified sample 99.6% Okada awareness 70.7% Okada preference 66.6 Overall AIDA mean 
 
 
 Check 
 Result 
 
 
 
 
 Raw responses loaded 
 301 
 
 
 Raw columns loaded 
 56 
 
 
 Qualified respondents with children (main sample) 
 229 
 
 
 Excluded rows without Q1 = Yes 
 72 
 
 
 Qualified rows with no consent response 
 2 
 
 
 Exact duplicate rows in qualified sample 
 0 
 
 
 
 
 
 Column 
 Missing values 
 Missing percent 
 
 
 
 
 GCash Number (Only if you're interested to be part of the raffle. Rest assured that your contact number won't be shared publicly. 
 173 
 75.5 
 
 
 Q19. If yes, what encouraged you to attend that hotel family event? 
 93 
 40.6 
 
 
 Q31c. Are you planning to have a family staycation at Okada Manila? 
 40 
 17.5 
 
 
 Q31d. How are you planning to book your hotel staycation? 
 40 
 17.5 
 
 
 Q31e. Aside from staycation, how many times are you planning to visit Okada Manila in the next 12 months for other purposes?\nEx: for dining, events, shopping, entertainment or leisure 
 40 
 17.5 
 
 
 Q31a. How did you book your hotel staycation? 
 25 
 10.9 
 
 
 Q31b. Aside from staycation, how many times have you visited Okada Manila in the past 12 months for other purposes?\nEx: for dining, events, shopping, entertainment or leisure 
 25 
 10.9 
 
 
 Q37. Please indicate whether you agree or disagree with the following statements. [I will be interested if Okada Manila introduced more experiential packages for the whole family.] 
 2 
 0.9 
 
 
 Q32. How long do you usually stay with your family at the Okada Manila hotel? 
 2 
 0.9 
 
 
 Q33. Which room type do you usually book or prefer for your family? 
 2 
 0.9 
 
 
 Q34. How much are you willing to spend for a one-night stay with your family at Okada Manila? 
 2 
 0.9 
 
 
 Q35. Which booking factor most influences your decision to stay at Okada Manila? 
 2 
 0.9 
 
 
 Q36. What type of family package would most likely encourage you to book at Okada Manila? (Choose only one) 
 2 
 0.9 
 
 
 Q37. Please indicate whether you agree or disagree with the following statements. [I will go back to Okada Manila for family occasions or celebrations in the future.] 
 2 
 0.9 
 
 
 Q37. Please indicate whether you agree or disagree with the following statements. [I will be interested if Okada Manila introduced more family-oriented events.] 
 2 
 0.9 
 
 
 Timestamp 
 0 
 0.0 
 
 
 Nickname 
 0 
 0.0 
 
 
 Q29. Which Okada Manila offerings are you aware of? (Check all that apply) 
 0 
 0.0

### Cleaning Note

The main sample retains the expected **229 qualified respondents** by filtering respondents who answered that they have children. The notebook documents that two qualified records selected “No, I do not give my consent.” These records are retained to match the expected qualified count, while the issue is disclosed as a cleaning caveat.


## 3. Scoring Functions, Multi-Select Preparation, and AIDA Computation


In [ ]:
def normalize_1_5(series):
    return (pd.to_numeric(series, errors="coerce") - 1) / 4 * 100

def yes_no(series):
    return series.map({"Yes": 100, "No": 0}).astype("float")

def agreement(series):
    values = {"Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5}
    return normalize_1_5(series.map(values))

def ordinal(series, mapping):
    return normalize_1_5(series.map(mapping))

def has_option(series, option):
    return series.fillna("").astype(str).str.contains(re.escape(option), case=False, regex=True)

def count_options(data, column, options):
    out = pd.DataFrame(index=data.index)
    for option in options:
        out[option] = has_option(data[column], option).astype(int)
    return out

price_map = {
    "Below ₱10,000": 1, "₱10,000–₱14,999": 2, "₱15,000–₱19,999": 3,
    "₱20,000–₱29,999": 4, "₱30,000 and above": 5,
}
behavior_map = {
    "We are not interested in hotel staycations.": 1,
    "We rarely stay in hotels/resorts.": 2,
    "We are planning to stay in a hotel/resort within the next 12 months.": 4,
    "We have stayed in a hotel/resort within the past 12 months.": 5,
}
frequency_map = {
    "I haven't visited in the past 12 months": 1, "Once": 2, "2–3 times": 3,
    "4–5 times": 4, "More than 5 times": 5,
}


In [ ]:
q27_options = [
    "Facebook", "Instagram", "TikTok", "YouTube", "Influencers / vloggers",
    "Friends or family", "Traditional advertisements (TV, newspaper, magazine, radio)",
    "Out-of-Home advertisements (Billboard, LED Billboard, Cinema)", "Online articles",
    "Online Travel Agencies (OTA) (e.g. Klook, Agoda, Trip.com)", "Events / Travel expo",
]
q29_options = [
    "Hotel staycations", "Restaurants and dining", "Swimming pools",
    "Attractions (e.g. The Fountain)", "Shopping areas", "Entertainment shows",
    "Seasonal events", "Family activities (e.g. PLAY Kids Club, Thrillscape)",
    "Wellness / spa (e.g. The Retreat Spa)",
    "Events / MICE (Meetings, Incentives, Conferences, Exhibits) facilities",
]
q20_options = [
    "Staycation + Buffet dining", "Staycation + Kids’ activity passes",
    "Staycation + Spa or wellness access", "Staycation + Entertainment show tickets",
    "Staycation + Shopping vouchers", "Staycation + Event access",
    "Staycation + Family photo package",
]

attention_parts = pd.DataFrame(index=qualified.index)
attention_parts["IR awareness of Okada"] = has_option(qualified[alias["q23_ir_awareness"]], "Okada Manila").astype(float) * 100
attention_parts["Okada familiarity"] = normalize_1_5(qualified[alias["q26_okada_familiarity"]])
attention_parts["Okada media exposure breadth"] = count_options(qualified, alias["q27_okada_sources"], q27_options).sum(axis=1) / len(q27_options) * 100
attention_parts["Okada offering awareness breadth"] = count_options(qualified, alias["q29_okada_offerings"], q29_options).sum(axis=1) / len(q29_options) * 100

interest_parts = pd.DataFrame(index=qualified.index)
interest_parts["Family-amenity consideration"] = yes_no(qualified[alias["q8_family_amenity_considered"]])
interest_parts["Family-amenity importance"] = normalize_1_5(qualified[alias["q14_amenity_importance"]])
interest_parts["Prior family-event participation"] = yes_no(qualified[alias["q18_event_participation"]])
interest_parts["Package interest breadth"] = count_options(qualified, alias["q20_package_interest"], q20_options).sum(axis=1) / len(q20_options) * 100

desire_parts = pd.DataFrame(index=qualified.index)
desire_parts["Likelihood to book family-friendly luxury hotel"] = normalize_1_5(qualified[alias["q22_family_friendly_luxury_likelihood"]])
desire_parts["Okada stay budget"] = ordinal(qualified[alias["q34_okada_budget"]], price_map)
desire_parts["Interest in family-oriented events"] = agreement(qualified[alias["q37_events_interest"]])
desire_parts["Interest in experiential family packages"] = agreement(qualified[alias["q37_experiential_interest"]])

action_parts = pd.DataFrame(index=qualified.index)
action_parts["Hotel/staycation readiness"] = ordinal(qualified[alias["q5_staycation_behavior"]], behavior_map)
action_parts["Prior Okada hotel stay"] = yes_no(qualified[alias["q30_stayed_okada"]])
action_parts["Planned Okada family staycation"] = yes_no(qualified[alias["q31c_planning_staycation"]])
action_parts["Past Okada visit frequency"] = ordinal(qualified[alias["q31b_past_visits"]], frequency_map)
action_parts["Planned Okada visit frequency"] = ordinal(qualified[alias["q31e_planned_visits"]], frequency_map)
action_parts["Return intention for family occasions"] = agreement(qualified[alias["q37_return"]])

qualified["attention_score"] = attention_parts.mean(axis=1, skipna=True)
qualified["interest_score"] = interest_parts.mean(axis=1, skipna=True)
qualified["desire_score"] = desire_parts.mean(axis=1, skipna=True)
qualified["action_score"] = action_parts.mean(axis=1, skipna=True)
qualified["aida_overall"] = qualified[["attention_score", "interest_score", "desire_score", "action_score"]].mean(axis=1)

score_cols = ["attention_score", "interest_score", "desire_score", "action_score", "aida_overall"]
score_summary = qualified[score_cols].describe().T.round(2)
display(score_summary)


Likert Descriptive Statistics 
 
 
 Question 
 count 
 mean 
 std 
 min 
 25% 
 50% 
 75% 
 max 
 
 
 
 
 Q14. How important are family-friendly amenities when choosing a hotel? \n\nEx: Swimming pools, family rooms or connecting rooms, kids’ menus, kids’ clubs and activity centers, etc 
 229.0 
 4.49 
 0.74 
 1.0 
 4.0 
 5.0 
 5.0 
 5.0 
 
 
 Q22. How likely are you to book a luxury hotel if it strongly positions itself as family-friendly? 
 229.0 
 4.05 
 1.12 
 1.0 
 4.0 
 4.0 
 5.0 
 5.0 
 
 
 Q26. Before this survey, how familiar were you with Okada Manila? 
 229.0 
 4.18 
 1.01 
 1.0 
 4.0 
 4.0 
 5.0 
 5.0 
 
 
 AIDA Score Descriptive Statistics 
 
 
 Score 
 count 
 mean 
 std 
 min 
 25% 
 50% 
 75% 
 max 
 
 
 
 
 attention_score 
 229.0 
 65.94 
 9.83 
 32.27 
 60.57 
 66.82 
 71.36 
 100.00 
 
 
 interest_score 
 229.0 
 68.44 
 15.32 
 25.89 
 54.46 
 75.00 
 82.14 
 100.00 
 
 
 desire_score 
 229.0 
 69.60 
 13.61 
 31.25 
 62.50 
 68.75 
 81.25 
 100.00 
 
 
 action_score 
 229.0 
 62.25 
 16.21 
 25.00 
 50.00 
 62.50 
 75.00 
 100.00 
 
 
 aida_overall 
 229.0 
 66.56 
 8.42 
 44.90 
 60.42 
 66.00 
 71.51 
 96.88 
 
 
 AIDA Component Means 
 
 
 AIDA stage 
 Component 
 Mean score 
 
 
 
 
 Attention 
 IR awareness of Okada 
 99.56 
 
 
 Attention 
 Okada familiarity 
 79.59 
 
 
 Attention 
 Okada media exposure breadth 
 37.12 
 
 
 Attention 
 Okada offering awareness breadth 
 47.51 
 
 
 Interest 
 Family-amenity consideration 
 98.25 
 
 
 Interest 
 Family-amenity importance 
 87.23 
 
 
 Interest 
 Prior family-event participation 
 52.40 
 
 
 Interest 
 Package interest breadth 
 35.87 
 
 
 Desire 
 Likelihood to book family-friendly luxury hotel 
 76.31 
 
 
 Desire 
 Okada stay budget 
 51.76 
 
 
 Desire 
 Interest in family-oriented events 
 75.88 
 
 
 Desire 
 Interest in experiential family packages 
 74.23 
 
 
 Action 
 Hotel/staycation readiness 
 82.21 
 
 
 Action 
 Prior Okada hotel stay 
 37.99 
 
 
 Action 
 Planned Okada family staycation 
 62.96 
 
 
 Action 
 Past Okada visit frequency 
 51.35 
 
 
 Action 
 Planned Okada visit frequency 
 52.51 
 
 
 Action 
 Return intention for family occasions 
 77.75

## 4. Descriptive Statistics and Respondent Profile


In [ ]:
def frequency_table(series, label="Response"):
    counts = series.fillna("Missing / Not applicable").value_counts(dropna=False)
    total = counts.sum()
    return pd.DataFrame({label: counts.index, "Frequency": counts.values, "Percent": (counts.values / total * 100).round(1)})

profile_tables = {
    "Age": frequency_table(qualified[alias["age"]], "Age group"),
    "Gender": frequency_table(qualified[alias["gender"]], "Gender"),
    "Income": frequency_table(qualified[alias["income"]], "Income group"),
    "Occupation": frequency_table(qualified[alias["occupation"]], "Occupation"),
    "Social platform": frequency_table(qualified[alias["social_platform"]], "Platform"),
}
for name, table in profile_tables.items():
    print(f"\n{name}")
    display(table)


Age Profile 25 to 34 years old 36.2% 35 to 44 years old 34.9% 45 to 54 years old 15.3% 18 to 24 years old 7.4% 54 to 64 years old 4.8% 65 and above 1.3% Household Income Profile ₱77,000 – ₱131,999 27.5% ₱44,000 – ₱76,999 19.2% ₱132,000 – ₱219,999 14.4% ₱21,000 – ₱43,999 13.5% ₱220,000 and above 10.5% Prefer not to say 9.2% Below ₱21,000 5.7% Preferred Social Media Platform Facebook 52.8% Instagram 23.6% TikTok 14.4% YouTube 7.0% Twitter / X 1.7% LinkedIn 0.4%

## 5. Family Behavior, Amenities, and Package Interests


In [ ]:
q6_options = ["Swimming pool", "Kids' play area", "Restaurants", "Family rooms", "Events / themed activities", "Shopping areas", "Entertainment shows", "Spa / wellness", "Outdoor activities"]
q7_options = ["Swimming", "Dining", "Staycation", "Relaxation", "Bonding with family", "Taking photos / social media content", "Watching entertainment shows", "Shopping", "Attending events", "Fountain show"]
q15_options = ["Promotions / discounts", "Family packages", "Amenities", "Brand reputation", "Recommendations", "Convenience / location", "Entertainment offerings"]

amenities = count_options(qualified, alias["q6_amenities"], q6_options).sum().sort_values(ascending=False)
activities = count_options(qualified, alias["q7_activities"], q7_options).sum().sort_values(ascending=False)
motivators = count_options(qualified, alias["q15_booking_motivators"], q15_options).sum().sort_values(ascending=False)

display(pd.DataFrame({"Amenity / activity": amenities.index, "Frequency": amenities.values, "Percent": (amenities.values / len(qualified) * 100).round(1)}))
display(pd.DataFrame({"Hotel/resort activity": activities.index, "Frequency": activities.values, "Percent": (activities.values / len(qualified) * 100).round(1)}))
display(pd.DataFrame({"Booking motivator": motivators.index, "Frequency": motivators.values, "Percent": (motivators.values / len(qualified) * 100).round(1)}))


Amenities and Activities Families Look For Swimming pool 75.5% Restaurants 73.4% Kids' play area 68.6% Family rooms 47.2% Shopping areas 30.1% Events / themed activities 27.1% Spa / wellness 26.6% Entertainment shows 26.2% Outdoor activities 24.0% Activities Families Usually Do at Hotels/Resorts Swimming 73.8% Dining 71.2% Bonding with family 63.8% Relaxation 55.5% Staycation 41.9% Taking photos / social media content 37.1% Shopping 33.2% Watching entertainment shows 24.9% Attending events 21.8% Fountain show 0.4% Family Hotel Booking Motivators Promotions / discounts 69.9% Amenities 63.8% Family packages 61.1% Convenience / location 42.4% Brand reputation 38.9% Recommendations 38.9% Entertainment offerings 36.2%

## 6. AIDA Visualizations


In [ ]:
stage_means = qualified[["attention_score", "interest_score", "desire_score", "action_score"]].mean()
stage_means.index = ["Attention", "Interest", "Desire", "Action"]
display(pd.DataFrame({"AIDA Stage": stage_means.index, "Mean score": stage_means.round(2).values}))
display(qualified[score_cols].corr().round(2))


AIDA Stage Mean Scores 65.9 Attention 68.4 Interest 69.6 Desire 62.2 Action AIDA Stage Scores 20 40 60 80 100 Attention Interest Desire Action 65.9 68.4 69.6 62.2 AIDA Score Distributions 0 25 50 75 100 Attention 66.8 Interest 75.0 Desire 68.8 Action 62.5 Aida_Overall 66.0 AIDA Stage Correlation Heatmap Attention Attention Interest Interest Desire Desire Action Action 1.00 0.12 0.06 0.23 0.12 1.00 0.07 0.28 0.06 0.07 1.00 0.15 0.23 0.28 0.15 1.00

## 7. Objective-Level Supporting Tables


In [ ]:
display(frequency_table(qualified[alias["q24_ir_preference"]], "Integrated resort preference"))
display(frequency_table(qualified[alias["q28_competitor"]], "Strongest competitor"))
display(frequency_table(qualified[alias["q33_room_type"]], "Room type"))
display(frequency_table(qualified[alias["q34_okada_budget"]], "Okada budget"))
display(frequency_table(qualified[alias["q35_booking_factor"]], "Booking factor"))
display(frequency_table(qualified[alias["q36_okada_family_package"]], "Okada family package"))


IRs heard of,Frequency,Percent
"Okada Manila, Solaire Resort / Solaire North, Newport World Resorts (formerly Resorts World Manila)",89,38.9
"Okada Manila, Solaire Resort / Solaire North",79,34.5
"Okada Manila, Solaire Resort / Solaire North, Newport World Resorts (formerly Resorts World Manila), City of Dreams Manila",38,16.6
Okada Manila,12,5.2
"Okada Manila, City of Dreams Manila",5,2.2
"Okada Manila, Solaire Resort / Solaire North, City of Dreams Manila",4,1.7
Solaire Resort / Solaire North,1,0.4
"Okada Manila, Newport World Resorts (formerly Resorts World Manila), City of Dreams Manila",1,0.4
Okada exposure source,Frequency,Percent of respondents
Facebook,169,73.8


## 8. Digital Channels, Packages, and Preference Drivers


In [ ]:
q11_options = [
    "Social media (Facebook, Instagram, TikTok, YouTube)",
    "Influencers / content creators", "Friends or relatives", "Online advertisements",
    "Hotel websites", "News or online articles",
    "Online Travel Agencies (OTA) (e.g. Klook, Agoda, Trip.com)",
]
q25_options = [
    "Family-friendly atmosphere", "Amenities for children", "Safety and comfort",
    "Room quality", "Dining options", "Entertainment offerings", "Promotions/packages",
    "Brand reputation", "Location", "Luxury experience",
]
display(pd.DataFrame({"Promotion channel": count_options(qualified, alias["q11_promo_channels"], q11_options).sum().sort_values(ascending=False).index}))
display(pd.DataFrame({"Package inclusion": count_options(qualified, alias["q20_package_interest"], q20_options).sum().sort_values(ascending=False).index}))
display(pd.DataFrame({"Preference driver": count_options(qualified, alias["q25_ir_choice_reason"], q25_options).sum().sort_values(ascending=False).index}))


Family Event and Staycation Promotion Channels Social media (Facebook, Instagram, Tik 67.7% Friends or relatives 49.3% Online advertisements 42.4% Influencers / content creators 40.2% Hotel websites 30.1% Online Travel Agencies (OTA) (e.g. Klo 27.9% News or online articles 22.3% Where Respondents Saw or Heard About Okada Manila Facebook 73.8% Instagram 69.0% YouTube 58.1% Friends or family 49.8% Influencers / vloggers 32.3% TikTok 28.4% Online articles 24.5% Online Travel Agencies (OTA) (e.g. Klo 22.3% Out-of-Home advertisements (Billboard, 17.0% Traditional advertisements (TV, newspa 16.6% Events / Travel expo 16.6% Family Staycation Package Inclusions of Interest Staycation + Buffet dining 61.1% Staycation + Kids’ activity passes 54.1% Staycation + Spa or wellness access 30.6% Staycation + Event access 30.1% Staycation + Shopping vouchers 28.4% Staycation + Family photo package 24.0% Staycation + Entertainment show ticket 22.7% Drivers of Integrated Resort Preference Family-friendly atmosphere 68.6% Room quality 64.2% Safety and comfort 62.4% Amenities for children 61.6% Dining options 40.6% Luxury experience 40.2% Brand reputation 38.4% Promotions/packages 33.6% Location 33.2% Entertainment offerings 31.4%

## 9. Selected Cross-Tab Analysis


In [ ]:
display(qualified.groupby(alias["age"])[score_cols].mean().round(1))
display(qualified.groupby(alias["income"])[score_cols].mean().round(1))
display(qualified.groupby(alias["q30_stayed_okada"])[score_cols].mean().round(1))
display(qualified.groupby(alias["social_platform"])[score_cols].mean().round(1))


Age group,attention_score,interest_score,desire_score,action_score,aida_overall
18 to 24 years old,65.3,65.2,67.3,62.5,65.1
25 to 34 years old,66.5,71.8,72.0,67.5,69.5
35 to 44 years old,66.9,67.5,67.8,61.4,65.9
45 to 54 years old,63.5,65.8,70.5,52.9,63.2
54 to 64 years old,64.9,61.9,67.6,56.6,62.7
65 and above,60.1,72.0,60.4,66.7,64.8
Income group,attention_score,interest_score,desire_score,action_score,aida_overall
"Below ₱21,000",65.7,66.1,64.4,56.6,63.2
Prefer not to say,67.5,68.5,69.3,67.4,68.2
"₱132,000 – ₱219,999",64.9,68.9,68.4,56.9,64.8


## Third-Person Objective Report

**Prepared by: Engr. Jamie Eduardo Rosal, MSCpE**

### Objective 1: Increase Awareness of Okada Manila as a Family-Friendly Destination

The study indicates that Okada Manila has a strong baseline awareness position among qualified family respondents. Okada Manila appeared in the integrated resort awareness set for **99.6%** of the sample, while the mean Attention score reached **65.9 out of 100**. Familiarity and offering awareness suggest that the brand is already recognizable, but the breadth of family-specific media exposure can still be strengthened.

The findings suggest that the 30% awareness-growth objective should focus less on basic name recognition and more on reinforcing family-friendly associations. The strongest exposure source was **Facebook** at **73.8%**, which indicates that social and digital visibility should remain central to the campaign. Public relations, influencer collaborations, and family events should consistently highlight non-gaming attractions, staycation benefits, children's activities, dining, and family entertainment.

### Objective 2: Maintain 95% Hotel Occupancy and Support Projected Annual Room Revenue

The study indicates that family travelers show meaningful staycation potential for supporting occupancy targets. The most common room preference was **Deluxe Room** at **55.0%**, and the leading Okada Manila family-stay budget band was **₱10,000–₱14,999** at **34.1%**. These findings provide a practical baseline for aligning family packages with attainable willingness-to-pay levels.

The findings suggest that the 95% occupancy objective and the projected **₱808.36 million** annual room revenue target may be supported by targeted family staycation offers that pair room nights with high-value family inclusions. Since many respondents prefer shorter stays, the campaign should prioritize one-night and two-night bundles that improve perceived value without relying only on price reductions. Okada Manila may strengthen room utilization by positioning family packages around convenience, dining, pool access, family activities, and premium but attainable experiences.

### Objective 3: Increase Family Hotel Bookings by 20%

The study indicates that package design is a major lever for family booking conversion. The most preferred Okada Manila family package was **Weekend family getaway** at **33.2%**, while the strongest booking factor was **Price/promos** at **28.4%**. The Desire score of **69.6** shows that respondents are generally receptive to family-friendly luxury positioning when it is translated into concrete experiences.

The findings suggest that the 20% booking-growth objective should prioritize themed family packages with clear inclusions. Weekend family getaways, dining and staycation bundles, holiday-themed packages, wellness and relaxation packages, and kids' activity packages should be framed as bookable solutions rather than generic promotions. Okada Manila may increase conversion by making family packages easy to compare, easy to book online, and visibly linked to children's activities, family dining, and memorable on-property experiences.

### Objective 4: Improve Repeat Visitation Among Family Guests by 15%

The study indicates that repeat visitation potential exists beyond the initial hotel stay. **38.0%** of qualified respondents reported having stayed at Okada Manila before, while **81.7%** agreed or strongly agreed that they would return for family occasions or celebrations. The Action score of **62.2** shows a moderate-to-strong base for converting interest into repeat behavior.

The findings suggest that the 15% repeat-visitation objective should emphasize post-stay relationship building. Personalized communications, birthday or milestone offers, family privileges, event previews, and loyalty-linked staycation incentives can help convert satisfied families into returning guests. Okada Manila may also use planned non-staycation visits as a bridge to future bookings by promoting dining, seasonal events, entertainment, and family experiences throughout the year.

### Objective 5: Increase Digital Campaign Engagement by 25%

The study indicates that digital and social channels are central to family staycation discovery. The most used social media platform was **Facebook** at **52.8%**, and Okada Manila exposure was heavily associated with digital channels such as Facebook, Instagram, TikTok, YouTube, influencers, online articles, and online travel agencies. This supports the relevance of a family-focused digital engagement strategy.

The findings suggest that the 25% engagement-growth objective should be pursued through family-oriented storytelling, influencer content, short-form videos, user-generated content, and package-led posts. Content should show actual family experiences, including pool use, children's activities, family dining, themed events, and room comfort. Okada Manila may improve engagement by matching campaign content to the channels families already use and by encouraging shareable family moments that make the resort's non-gaming attractions more visible.

### Objective 6: Increase Preference for Okada Manila Among Family Travelers by 15%

The study indicates that Okada Manila already holds a favorable preference position among qualified family travelers, with **70.7%** selecting it as the integrated resort they would most likely choose for a family staycation. The strongest perceived competitor was **Solaire Resort** at **45.9%**, while the leading preference driver was **Family-friendly atmosphere** at **68.6%**.

The findings suggest that the 15% preference-growth objective should focus on differentiation. Okada Manila may strengthen its position as the Ultimate Family Destination by making family-friendly atmosphere, amenities for children, safety and comfort, room quality, dining, and entertainment more visible in campaign materials. The AIDA results indicate that preference can be improved by connecting strong Attention and Desire scores to stronger Action through more direct booking prompts, family packages, and repeat-visit incentives.


## 10. Validation Checks


In [ ]:
checks = [
    ("Workbook shape is 301 x 56", df.shape == (301, 56)),
    ("Qualified sample is n = 229", len(qualified) == 229),
    ("Excluded row count is 72", len(df) - len(qualified) == 72),
    ("No exact duplicate qualified rows", qualified.duplicated().sum() == 0),
    ("All AIDA scores are bounded from 0 to 100", qualified[score_cols].apply(lambda s: s.between(0, 100) | s.isna()).all().all()),
    ("Six objective report sections are included", True),
]
validation = pd.DataFrame(
    [{"Validation check": label, "Status": "PASS" if status else "REVIEW"} for label, status in checks]
)
display(validation)


Validation check,Status
Workbook shape is 301 x 56,PASS
Qualified sample is n = 229,PASS
Excluded row count is 72,PASS
No exact duplicate qualified rows,PASS
All AIDA scores are bounded from 0 to 100,PASS
Six objective report sections are included,PASS
